In [11]:
import torch
from torch import nn
from torch.utils.data import DataLoader, random_split
from torchvision import models, datasets, transforms


In [ ]:
BATCH_SIZE = 64
EPOCHS = 10
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4

IMAGE_SIZE = 256

generator = torch.Generator().manual_seed(42)

# select device based on machine
device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

print("Device:", device)

Device: cpu


In [13]:
# need to upscale images, and create 3 channel RGB images for convnext model
transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)), # upscale image
    transforms.Grayscale(num_output_channels=3), # convert 1 channel to 3 channels
    transforms.ToTensor(),
])

# download MNIST train dataset and transform it
full_train = datasets.MNIST(
    root="./data", train=True, download=True, transform=transform
)
# download MNIST test dataset and transform it
test_dataset = datasets.MNIST(
    root="./data", train=False, download=True, transform=transform
)

# split training set into 80% train, and 20% validation
train_dataset, val_dataset = random_split(full_train, [0.8, 0.2], generator=generator)

# dataloaders for each dataset, train dataset shuffles each epoch
train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0,
    generator=torch.Generator().manual_seed(42),
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0
)
test_loader = DataLoader(
    test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0
)


print("Train:", len(train_dataset))
print("Validation:", len(val_dataset))
print("Test:", len(test_dataset))

Train: 48000
Validation: 12000
Test: 10000


In [14]:
# initalise ConvNeXt model with random weights
model = models.convnext_large(weights=None)

# get the input features of the final layer
num_features = model.classifier[2].in_features

# replace the final layer with only 10 output classes
num_classes = 10
model.classifier[2] = nn.Linear(num_features, num_classes)

# move to GPU
model = model.to(device)

# define loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

In [15]:
# training loop
for epoch in range(EPOCHS):
    model.train()

    running_loss = 0.0
    correct_train = 0
    total_train = 0

    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad() # zero the parameter gradients
        outputs = model(inputs) # forward pass through network
        loss = criterion(outputs, labels) # calculate loss
        loss.backward() # compute gradients
        optimizer.step() # update weights

        running_loss += loss.item() * inputs.size(0)

    epoch_loss = running_loss / len(train_dataset)
    print(f"Epoch {epoch+1}/{EPOCHS} - Loss: {epoch_loss:.4f}")


KeyboardInterrupt: 